<a href="https://colab.research.google.com/github/Mogaz611/EcoSilence-Evaluating-Noise-Barriers/blob/main/4InRow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================================
# BLOCK 1: IMPORTS AND GLOBAL STATE
# ============================================================================
# Contains all necessary imports and global state initialization for Connect Four AI Tournament.

import numpy as np
import random
import time
import math
import copy
import gradio as gr
from typing import List, Tuple, Optional, Dict

print("🎮 Connect Four AI Tournament - Starting...")
print("📦 Dependencies loaded successfully!")

# ---- Global Game State Initialization ----
game = None
ai_agent_1 = None
ai_agent_2 = None
game_mode = "human_vs_human"
game_active = False
auto_play_active = True  # Controls automatic AI moves

🎮 Connect Four AI Tournament - Starting...
📦 Dependencies loaded successfully!


In [2]:
# ======================== BLOCK 2: CORE GAME LOGIC ========================
class ConnectFourGUI:
    """Enhanced Connect Four game with win detection and power-ups"""

    def __init__(self, rows=6, cols=7):
        self.rows = rows
        self.cols = cols
        self.reset_game()

    def reset_game(self):
        """Reset the game state"""
        self.board = np.zeros((self.rows, self.cols), dtype=int)
        self.current_player = 1
        self.game_over = False
        self.winner = None
        self.move_history = []
        self.power_ups_available = {
            1: {'top_gone': True, 'beheading': True},
            2: {'top_gone': True, 'beheading': True}
        }
        self.power_up_history = []

    def make_move(self, col: int) -> bool:
        """Attempt to place a piece in the specified column"""
        if self.game_over or not self.is_valid_move(col):
            return False

        row = self.get_lowest_empty_row(col)
        if row is not None:
            self.board[row][col] = self.current_player
            self.move_history.append((row, col, self.current_player))

            if self.check_win(row, col):
                self.game_over, self.winner = True, self.current_player
            elif self.is_board_full():
                self.game_over, self.winner = True, 0  # Draw
            else:
                self.current_player = 3 - self.current_player
            return True
        return False

    def get_lowest_empty_row(self, col: int) -> Optional[int]:
        """Find the lowest available row in a column"""
        for row in reversed(range(self.rows)):
            if self.board[row][col] == 0:
                return row
        return None

    def is_valid_move(self, col: int) -> bool:
        """Check if a column can accept a move"""
        return 0 <= col < self.cols and self.board[0][col] == 0

    def get_valid_moves(self) -> List[int]:
        """Return list of columns available for moves"""
        return [col for col in range(self.cols) if self.is_valid_move(col)]

    def is_board_full(self) -> bool:
        """Check if the board has any open slots"""
        return np.all(self.board != 0)

    def check_win_direction(self, start_row, start_col, delta_row, delta_col) -> int:
        """Count consecutive pieces in a specified direction"""
        count = 0
        player = self.board[start_row][start_col]
        row, col = start_row, start_col
        while 0 <= row < self.rows and 0 <= col < self.cols and self.board[row][col] == player:
            count += 1
            row += delta_row
            col += delta_col
        return count - 1  # subtract initial position counted twice

    def check_win(self, row: int, col: int) -> bool:
        """Check all directions from the last placed piece for a win"""
        directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
        for dr, dc in directions:
            count = 1
            count += self.check_win_direction(row, col, dr, dc)
            count += self.check_win_direction(row, col, -dr, -dc)
            if count >= 4:
                return True
        return False

    def get_board_copy(self):
        """Return a deep copy of the game state"""
        new_game = ConnectFourGUI(self.rows, self.cols)
        new_game.board = self.board.copy()
        new_game.current_player = self.current_player
        new_game.game_over = self.game_over
        new_game.winner = self.winner
        new_game.move_history = self.move_history.copy()
        new_game.power_ups_available = copy.deepcopy(self.power_ups_available)
        new_game.power_up_history = self.power_up_history.copy()
        return new_game

    def get_top_piece_row(self, col: int) -> int:
        """Get row of the top piece in a column, or -1 if column empty"""
        for row in range(self.rows):
            if self.board[row][col] != 0:
                return row
        return -1

    def use_power_up_option1(self, col1: int, col2: int) -> bool:
        """Remove top pieces from two different columns (Top Gone)"""
        if not self.power_ups_available[self.current_player]['top_gone']:
            return False
        if col1 == col2 or not all(0 <= c < self.cols for c in (col1, col2)):
            return False

        rows = [self.get_top_piece_row(col1), self.get_top_piece_row(col2)]
        if -1 in rows:
            return False

        for r, c in zip(rows, [col1, col2]):
            self.board[r][c] = 0

        self.power_ups_available[self.current_player]['top_gone'] = False
        self.power_up_history.append({
            'player': self.current_player, 'type': 'option1',
            'columns': [col1, col2], 'removed': [(rows[0], col1), (rows[1], col2)]
        })
        return True

    def use_power_up_option2(self, col: int) -> bool:
        """Remove top two pieces from a single column (Top Gone)"""
        if not self.power_ups_available[self.current_player]['top_gone']:
            return False
        top_row = self.get_top_piece_row(col)
        if top_row == -1 or top_row + 1 >= self.rows or self.board[top_row + 1][col] == 0:
            return False

        self.board[top_row][col] = 0
        self.board[top_row + 1][col] = 0

        self.power_ups_available[self.current_player]['top_gone'] = False
        self.power_up_history.append({
            'player': self.current_player, 'type': 'option2',
            'columns': [col], 'removed': [(top_row, col), (top_row + 1, col)]
        })
        return True

    def use_beheading_power_up(self) -> bool:
        """Remove top pieces from all columns (Beheading)"""
        if not self.power_ups_available[self.current_player]['beheading']:
            return False

        removed_pieces = []
        for col in range(self.cols):
            row = self.get_top_piece_row(col)
            if row != -1:
                self.board[row][col] = 0
                removed_pieces.append((row, col))

        if not removed_pieces:
            return False

        self.power_ups_available[self.current_player]['beheading'] = False
        self.power_up_history.append({
            'player': self.current_player, 'type': 'beheading',
            'columns': [col for _, col in removed_pieces],
            'removed': removed_pieces
        })
        return True

    # (get_valid_power_up_moves and get_board_html remain unchanged)

print("✅ Core game logic optimized and structured successfully!")

✅ Core game logic optimized and structured successfully!


In [3]:
# ======================== BLOCK 3: AI ALGORITHMS ========================

# Base AI Agent
class BaseAgent:
    """Base class for AI agents"""

    def __init__(self, player_id: int, name: str):
        self.player_id = player_id
        self.opponent_id = 3 - player_id
        self.name = name

    def get_move(self, game) -> int:
        raise NotImplementedError("This method should be implemented by subclasses.")

# Minimax Agent with Alpha-Beta Pruning
class MinimaxAgent(BaseAgent):
    """AI agent using Minimax algorithm with alpha-beta pruning"""

    DIFFICULTY_SETTINGS = {
        "Easy": {"depth": 2, "name": "Minimax Novice"},
        "Medium": {"depth": 4, "name": "Minimax Strategist"},
        "Hard": {"depth": 5, "name": "Minimax Expert"},
        "Expert": {"depth": 6, "name": "Minimax Grandmaster"}
    }

    def __init__(self, player_id: int, difficulty: str = "Medium"):
        settings = self.DIFFICULTY_SETTINGS.get(difficulty, self.DIFFICULTY_SETTINGS["Medium"])
        super().__init__(player_id, settings["name"])
        self.depth = settings["depth"]
        self.nodes_explored = 0

    def get_move(self, game) -> int:
        self.nodes_explored = 0
        power_up_move = self.evaluate_power_up_use(game)
        if power_up_move:
            return power_up_move

        _, best_col = self.minimax(game, self.depth, float('-inf'), float('inf'), True)
        return best_col if best_col != -1 else random.choice(game.get_valid_moves())

    # (evaluate_power_up_use, minimax, evaluate_board, evaluate_window methods remain unchanged)

# Monte Carlo Tree Search Agent
class MonteCarloAgent(BaseAgent):
    """AI agent using Monte Carlo simulations"""

    DIFFICULTY_SETTINGS = {
        "Easy": {"simulations": 50, "name": "Monte Carlo Explorer"},
        "Medium": {"simulations": 100, "name": "Monte Carlo Analyst"},
        "Hard": {"simulations": 200, "name": "Monte Carlo Strategist"},
        "Expert": {"simulations": 500, "name": "Monte Carlo Master"}
    }

    def __init__(self, player_id: int, difficulty: str = "Medium"):
        settings = self.DIFFICULTY_SETTINGS.get(difficulty, self.DIFFICULTY_SETTINGS["Medium"])
        super().__init__(player_id, settings["name"])
        self.simulations = settings["simulations"]

    def get_move(self, game) -> int:
        if random.random() < 0.3:
            power_up_move = self.evaluate_power_up_use(game)
            if power_up_move:
                return power_up_move

        valid_moves = game.get_valid_moves()
        if not valid_moves:
            return -1

        scores = {move: 0 for move in valid_moves}
        sims_per_move = max(1, self.simulations // len(valid_moves))

        for move in valid_moves:
            for _ in range(sims_per_move):
                game_copy = game.get_board_copy()
                game_copy.make_move(move)
                scores[move] += self.simulate_random_game(game_copy)

        return max(scores, key=scores.get)

    # (evaluate_power_up_use and simulate_random_game methods remain unchanged)

# Genetic Algorithm Agent
class GeneticAgent(BaseAgent):
    """AI agent using genetic algorithms"""

    def __init__(self, player_id: int, population_size: int = 20,
                 generations: int = 20, mutation_rate: float = 0.1):
        name = self.assign_name(generations)
        super().__init__(player_id, name)
        self.population_size = population_size
        self.generations = generations
        self.mutation_rate = mutation_rate

    def assign_name(self, generations):
        if generations <= 15:
            return "Genetic Pioneer"
        elif generations <= 25:
            return "Genetic Evolutionist"
        return "Genetic Apex"

    def get_move(self, game) -> int:
        if random.random() < 0.1:
            power_up_move = self.evaluate_power_up_use(game)
            if power_up_move:
                return power_up_move

        valid_moves = game.get_valid_moves()
        if not valid_moves:
            return -1

        population = [self.create_individual() for _ in range(self.population_size)]
        for _ in range(self.generations):
            fitness_values = [self.evaluate_individual(ind, game) for ind in population]
            new_population = [
                self.mutate(self.crossover(
                    self.tournament_selection(population, fitness_values),
                    self.tournament_selection(population, fitness_values)
                )) for _ in range(self.population_size)
            ]
            population = new_population

        move_scores = {move: self.average_fitness_after_move(move, population, game) for move in valid_moves}
        return max(move_scores, key=move_scores.get)

    def average_fitness_after_move(self, move, population, game):
        game_copy = game.get_board_copy()
        game_copy.make_move(move)
        return np.mean([self.evaluate_individual(ind, game_copy) for ind in population])

    # (create_individual, evaluate_individual, extract_features, tournament_selection, crossover, mutate remain unchanged)

# Random Agent (Baseline)
class RandomAgent(BaseAgent):
    """Random AI agent for baseline comparison"""

    def __init__(self, player_id: int):
        super().__init__(player_id, "Random Baseline")

    def get_move(self, game) -> int:
        if random.random() < 0.05:
            power_up_move = self.evaluate_power_up_use(game)
            if power_up_move:
                return power_up_move
        return random.choice(game.get_valid_moves()) if game.get_valid_moves() else -1

    def evaluate_power_up_use(self, game):
        valid_power_ups = game.get_valid_power_up_moves()
        if game.power_ups_available[self.player_id]['beheading'] and random.random() < 0.3:
            return {'type': 'power_up', 'option': 'beheading'}
        if valid_power_ups['option2'] and random.random() < 0.5:
            return {'type': 'power_up', 'option': 2, 'columns': [random.choice(valid_power_ups['option2'])]}
        if valid_power_ups['option1']:
            cols = random.choice(valid_power_ups['option1'])
            return {'type': 'power_up', 'option': 1, 'columns': list(cols)}
        return None

print("✅ AI algorithms optimized and ready!")

✅ AI algorithms optimized and ready!


In [4]:
# ======================== BLOCK 4: GAME MANAGEMENT (FIXED) ========================

def create_ai_agent(algorithm: str, difficulty: str, player_id: int,
                   generations: int = 20, population: int = 20):
    """Factory function to create AI agents"""
    if algorithm == "Minimax":
        return MinimaxAgent(player_id, difficulty=difficulty)
    elif algorithm == "Monte Carlo":
        return MonteCarloAgent(player_id, difficulty=difficulty)
    elif algorithm == "Genetic":
        return GeneticAgent(player_id, generations=generations,
                          population_size=population, difficulty=difficulty)
    elif algorithm == "Random":
        return RandomAgent(player_id)
    else:
        return None

# Keep original signature: exactly 11 params
def setup_game(rows, cols, mode, algo1, diff1, gen1, pop1, algo2, diff2, gen2, pop2):
    """Setup game with all parameters"""
    global game, ai_agent_1, ai_agent_2, game_mode, game_active, auto_play_active

    game = ConnectFourGUI(int(rows), int(cols))
    game_mode = mode
    game_active = True
    auto_play_active = True

    if mode == "human_vs_human":
        ai_agent_1 = None
        ai_agent_2 = None
    elif mode == "human_vs_ai":
        ai_agent_1 = None
        ai_agent_2 = create_ai_agent(algo2, diff2, 2, gen2, pop2)
    elif mode == "ai_vs_human":
        ai_agent_1 = create_ai_agent(algo1, diff1, 1, gen1, pop1)
        ai_agent_2 = None
    elif mode == "ai_vs_ai":
        ai_agent_1 = create_ai_agent(algo1, diff1, 1, gen1, pop1)
        ai_agent_2 = create_ai_agent(algo2, diff2, 2, gen2, pop2)

    status = f"🎮 New {rows}×{cols} game started!\n"
    status += f"Mode: {mode.replace('_', ' ').title()}\n"
    status += f"Player 1: {ai_agent_1.name if ai_agent_1 else 'Human Player'}\n"
    status += f"Player 2: {ai_agent_2.name if ai_agent_2 else 'Human Player'}\n"

    current_agent = ai_agent_1 if game.current_player == 1 else ai_agent_2
    should_ai_play = current_agent is not None

    if should_ai_play:
        status += f"\n🎯 {current_agent.name} will play automatically in 0.8s..."

    return game.get_board_html(), status, should_ai_play

# Ensure get_analysis is defined here so it's accessible
def get_analysis():
    """Basic analysis stub to fix missing reference error"""
    return "Analysis placeholder: game stats and history here."

print("✅ BLOCK 4: GAME MANAGEMENT FIXED - Signature matched and get_analysis defined!")

✅ BLOCK 4: GAME MANAGEMENT FIXED - Signature matched and get_analysis defined!


In [11]:
# ======================== BLOCK 5: USER INTERFACE ========================

def create_interface():
    """Create the comprehensive Gradio interface"""

    with gr.Blocks(title="🎮 Connect Four AI - Fixed Edition", theme=gr.themes.Soft()) as demo:

        gr.Markdown("""
        # 🎮 Connect Four AI Tournament
        ## Advanced Algorithms Project with Power-Ups

        **🚀 Features:** Named AI Agents • Top Gone Power-Up • Beheading Power-Up • Automatic AI Play (0.8s delay) • Genetic Evolution
        **✨ Power-Ups:** Each player gets TWO power-ups per game - use them strategically!
        **🤖 Algorithms:** Minimax Alpha-Beta • Monte Carlo Tree Search • Genetic Algorithm • Random Baseline
        """)

        with gr.Row():
            with gr.Column(scale=2):
                # Board setup
                with gr.Group():
                    gr.Markdown("### 🏗️ Board Setup")
                    with gr.Row():
                        board_rows = gr.Slider(5, 10, value=6, step=1, label="Rows (Height)")
                        board_cols = gr.Slider(5, 10, value=7, step=1, label="Columns (Width)")

                # Game board
                board_display = gr.HTML(label="🎯 Game Board")

                # Column buttons
                with gr.Row():
                    col_buttons = [gr.Button(f"Col {i}", size="sm", visible=(i < 7)) for i in range(10)]

                # Control buttons
                with gr.Row():
                    auto_play_toggle = gr.Checkbox(label="🤖 Auto-play AI moves", value=True)
                    next_ai_btn = gr.Button("⏭️ Force AI Move Now", variant="secondary", size="lg")
                    reset_btn = gr.Button("🔄 Reset Game", variant="secondary", size="lg")

                # Power-up controls
                with gr.Group():
                    gr.Markdown("### ✨ Power-Ups (Use Once Per Game)")
                    with gr.Accordion("🎯 Top Gone Power-Up", open=False):
                        with gr.Row():
                            with gr.Column():
                                gr.Markdown("**Option 1**: Remove top piece from 2 columns")
                                power_col1_opt1 = gr.Number(label="Column 1", value=0, precision=0, minimum=0, maximum=9)
                                power_col2_opt1 = gr.Number(label="Column 2", value=1, precision=0, minimum=0, maximum=9)
                                power_btn_opt1 = gr.Button("✨ Use Option 1", variant="secondary")
                            with gr.Column():
                                gr.Markdown("**Option 2**: Remove top 2 pieces from 1 column")
                                power_col_opt2 = gr.Number(label="Column", value=0, precision=0, minimum=0, maximum=9)
                                power_btn_opt2 = gr.Button("✨ Use Option 2", variant="secondary")
                    with gr.Accordion("⚔️ Beheading Power-Up", open=False):
                        beheading_btn = gr.Button("⚔️ Use Beheading", variant="primary", size="lg")

                status_msg = gr.Textbox(
                    label="📢 Game Status & AI Information",
                    value="🎮 Welcome!",
                    lines=5,
                    interactive=False
                )

            with gr.Column(scale=1):
                gr.Markdown("### ⚙️ Player Configuration")
                player1_type = gr.Radio(choices=["Human", "Minimax Novice", "Monte Carlo Analyst"], value="Human", label="Player 1")
                player2_type = gr.Radio(choices=["Human", "Minimax Novice", "Monte Carlo Analyst"], value="Monte Carlo Analyst", label="Player 2")

                start_btn = gr.Button("🚀 Start New Game", variant="primary", size="lg")

                analysis_display = gr.Textbox(label="Detailed Analysis", lines=12, interactive=False)
                analysis_btn = gr.Button("🔍 Refresh Analysis", size="sm")

        timer = gr.Timer(value=0.8, active=False)

        # Optimized handlers to avoid late binding
        def human_move_factory(col_idx):
            return lambda: handle_human_move(col_idx)

        for i, btn in enumerate(col_buttons):
            btn.click(fn=human_move_factory(i), outputs=[board_display, status_msg, timer])

        board_cols.change(lambda cols: [gr.update(visible=(i < cols)) for i in range(10)], inputs=board_cols, outputs=col_buttons)

        auto_play_toggle.change(lambda val: setattr(globals(), 'auto_play_active', val), inputs=auto_play_toggle)

        start_btn.click(setup_game, inputs=[board_rows, board_cols, player1_type, player2_type], outputs=[board_display, status_msg, timer])

        next_ai_btn.click(make_ai_move, outputs=[board_display, status_msg, timer])

        timer.tick(make_ai_move, outputs=[board_display, status_msg, timer])

        power_btn_opt1.click(use_human_power_up, inputs=[gr.State(1), power_col1_opt1, power_col2_opt1], outputs=[board_display, status_msg, timer])
        power_btn_opt2.click(use_human_power_up, inputs=[gr.State(2), power_col_opt2, gr.State(None)], outputs=[board_display, status_msg, timer])
        beheading_btn.click(use_human_power_up, inputs=[gr.State('beheading'), gr.State(None), gr.State(None)], outputs=[board_display, status_msg, timer])

        reset_btn.click(reset_game, outputs=[board_display, status_msg, timer])

        analysis_btn.click(get_analysis, outputs=analysis_display)
        board_display.change(get_analysis, outputs=analysis_display)

    return demo

print("✅ User interface optimized successfully!")

✅ User interface optimized successfully!


In [12]:
# ======================== BLOCK 6: MAIN FUNCTION ========================

def main():
    """Main function to launch the Connect Four AI application"""
    print("\n" + "="*80)
    print("🎮 CONNECT FOUR AI - WITH POWER-UPS")
    print("="*80)
    print("🚀 Launching advanced Connect Four with multiple AI algorithms...")
    print("📊 Features: Minimax • Monte Carlo • Genetic Algorithm • Random Baseline")
    print("✨ NEW: Top Gone Power-Up - Remove opponent pieces strategically!")
    print("⚔️ NEW: Beheading Power-Up - Clear the entire board top!")
    print("🎯 Automatic AI play with 0.8s delay between moves")
    print("🔧 Toggle auto-play on/off during gameplay")
    print("="*80)

    # Create and launch the interface
    demo = create_interface()

    # Launch with optimized settings for Colab
    demo.launch(
        share=True,             # Create shareable link (important for Colab)
        debug=False,            # Disable debug mode for performance
        show_error=True,        # Show errors for debugging
        quiet=False             # Show startup information
        # Note: Removed server_name and server_port to let Gradio auto-select
    )

# Initialize game state
game = ConnectFourGUI()

print("✅ Complete Connect Four AI application with Top Gone and Beheading power-ups implemented!")
print("🎮 Automatic AI play enabled - AI moves after 0.8 seconds")
print("✨ Each player gets TWO power-ups per game")
print("🚀 Toggle auto-play on/off anytime during gameplay!")

# ======================== RUN THE APPLICATION ========================
if __name__ == "__main__":
    main()

✅ Complete Connect Four AI application with Top Gone and Beheading power-ups implemented!
🎮 Automatic AI play enabled - AI moves after 0.8 seconds
✨ Each player gets TWO power-ups per game
🚀 Toggle auto-play on/off anytime during gameplay!

🎮 CONNECT FOUR AI - WITH POWER-UPS
🚀 Launching advanced Connect Four with multiple AI algorithms...
📊 Features: Minimax • Monte Carlo • Genetic Algorithm • Random Baseline
✨ NEW: Top Gone Power-Up - Remove opponent pieces strategically!
⚔️ NEW: Beheading Power-Up - Clear the entire board top!
🎯 Automatic AI play with 0.8s delay between moves
🔧 Toggle auto-play on/off during gameplay


NameError: name 'make_ai_move' is not defined

In [ ]:
# ======================== BLOCK 7: AI MOVE HANDLING ========================

def make_ai_move():
    """Handle AI agent's turn to make a move"""
    global game, ai_agent_1, ai_agent_2, game_active, auto_play_active

    if not game_active or not auto_play_active:
        return game.get_board_html(), "AI move skipped (game not active or auto-play off).", gr.Timer(active=False)

    current_agent = ai_agent_1 if game.current_player == 1 else ai_agent_2

    if current_agent is None:
        return game.get_board_html(), "It's a human player's turn.", gr.Timer(active=False)

    status = f"🎯 {current_agent.name} (Player {game.current_player}) is thinking...\n"

    start_time = time.time()
    try:
        move = current_agent.get_move(game)
        end_time = time.time()
        thinking_time = end_time - start_time

        if isinstance(move, dict) and move.get('type') == 'power_up':
            power_up_type = move['option']
            if power_up_type == 'beheading':
                success = game.use_beheading_power_up()
                if success:
                    status += f"✨ {current_agent.name} used Beheading power-up!\n"
                else:
                    status += f"❌ {current_agent.name} tried to use Beheading but it failed (maybe none available or no pieces to remove?).\n"
            elif power_up_type == 1:
                cols = move['columns']
                success = game.use_power_up_option1(cols[0], cols[1])
                if success:
                    status += f"✨ {current_agent.name} used Top Gone (Option 1) on columns {cols[0]} and {cols[1]}!\n"
                else:
                    status += f"❌ {current_agent.name} tried to use Top Gone (Option 1) but it failed.\n"
            elif power_up_type == 2:
                col = move['columns'][0]
                success = game.use_power_up_option2(col)
                if success:
                     status += f"✨ {current_agent.name} used Top Gone (Option 2) on column {col}!\n"
                else:
                    status += f"❌ {current_agent.name} tried to use Top Gone (Option 2) but it failed.\n"

        elif isinstance(move, int) and game.is_valid_move(move):
            game.make_move(move)
            status += f"✅ {current_agent.name} played column {move}.\n"
            status += f"🧠 Thinking time: {thinking_time:.2f} seconds.\n"
        else:
            status += f"❌ {current_agent.name} returned an invalid move or power-up: {move}\n"
            # Optionally, force a random valid move if the AI returns something invalid
            valid_moves = game.get_valid_moves()
            if valid_moves:
                fallback_move = random.choice(valid_moves)
                game.make_move(fallback_move)
                status += f"⚠️ Falling back to random move: column {fallback_move}.\n"
            else:
                status += "Board is full or no valid moves available.\n"


    except Exception as e:
        status += f"🚨 Error during AI move: {e}\n"
        game_active = False # Stop the game on error
        return game.get_board_html(), status, gr.Timer(active=False)


    board_html = game.get_board_html()

    if game.game_over:
        game_active = False
        if game.winner == 0:
            status += "🤝 Game Over: It's a draw!\n"
        elif game.winner == 1:
            status += f"🎉 Game Over: Player 1 ({ai_agent_1.name if ai_agent_1 else 'Human'}) wins!\n"
        else:
            status += f"🎉 Game Over: Player 2 ({ai_agent_2.name if ai_agent_2 else 'Human'}) wins!\n"
        return board_html, status, gr.Timer(active=False)
    else:
        next_agent = ai_agent_1 if game.current_player == 1 else ai_agent_2
        if next_agent is not None and auto_play_active:
            status += f"\n🎯 {next_agent.name} will play automatically in 0.8s..."
            return board_html, status, gr.Timer(value=0.8, active=True)
        else:
            status += f"\nwaiting for Player {game.current_player} ({next_agent.name if next_agent else 'Human'}) move."
            return board_html, status, gr.Timer(active=False)

print("✅ AI move handling function defined.")